# 10 · Churn Evaluation v2 (honest evaluation)

**Purpose:** re-evaluate churn prediction with meaningful metrics and a no-skill baseline.
Notebooks 08 and 09 are kept unchanged as *Experiment 1 (baseline experiments)*.

**Method (leakage-safe):** features are built only from orders on or before the cutoff date (31-Dec-2019).
Label `churn = 1` if the customer placed **no** order after the cutoff, otherwise `0`.

**Metrics used:** ROC-AUC and PR-AUC (threshold-free), balanced accuracy, retained-customer recall.
Plain accuracy/F1 on the majority class is misleading here because ~81% of customers are churned.

In [1]:
# ---- CHANGE THESE PATHS IF YOUR FOLDERS ARE DIFFERENT -------------------------
ORDERS_PATH = "../data/processed/orders_cleaned.csv"   # cleaned orders file
TABLE_DIR   = "../outputs/tables"                    # where result tables are saved
CUTOFF      = "2019-12-31"
RANDOM_STATE = 42
# -------------------------------------------------------------------------------

import os, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, balanced_accuracy_score,
                             accuracy_score, f1_score, recall_score)

os.makedirs(TABLE_DIR, exist_ok=True)

## 1. Build customer features at the cutoff date

In [2]:
orders = pd.read_csv(ORDERS_PATH)
orders["order_date"] = pd.to_datetime(orders["order_date"])
cutoff = pd.Timestamp(CUTOFF)

history = orders[orders["order_date"] <= cutoff]     # used for features
future  = orders[orders["order_date"] >  cutoff]     # used ONLY for the label

cust = history.groupby("user_id").agg(
    total_orders=("user_id", "count"),
    total_spend=("sales_amount", "sum"),
    total_quantity=("sales_qty", "sum"),
    first_order=("order_date", "min"),
    last_order=("order_date", "max"),
).reset_index()

cust["average_order_value"] = cust["total_spend"] / cust["total_orders"]
cust["recency"] = (cutoff - cust["last_order"]).dt.days
cust["tenure_days"] = (cust["last_order"] - cust["first_order"]).dt.days
cust["active_months"] = (cust["tenure_days"] / 30).round(2)
cust["average_days_between_orders"] = np.where(cust["total_orders"] > 1,
                                               cust["tenure_days"] / (cust["total_orders"] - 1), 0)
cust["orders_per_month"] = np.where(cust["active_months"] > 0,
                                    cust["total_orders"] / cust["active_months"], cust["total_orders"])

cust["churn"] = (~cust["user_id"].isin(set(future["user_id"]))).astype(int)

FEATURES = ["total_orders", "total_spend", "total_quantity", "average_order_value", "recency",
            "tenure_days", "active_months", "average_days_between_orders", "orders_per_month"]
X, y = cust[FEATURES], cust["churn"]

print("Customers:", len(cust))
print("Churn rate:", round(y.mean(), 4), "| Retained rate:", round(1 - y.mean(), 4))

Customers: 72069
Churn rate: 0.8088 | Retained rate: 0.1912


## 2. Does any feature relate to churn? (diagnostic)
If churn is ~81% in every bucket, the features carry no signal.

In [3]:
def churn_by_bucket(series, bins, labels, name):
    t = (cust.groupby(pd.cut(series, bins=bins, labels=labels), observed=True)["churn"]
              .agg(churn_rate="mean", customers="count").reset_index())
    t.columns = ["bucket", "churn_rate", "customers"]
    t.insert(0, "feature", name)
    t["churn_rate"] = t["churn_rate"].round(4)
    return t

diag = pd.concat([
    churn_by_bucket(cust["recency"], [-1, 90, 180, 365, 730, 5000], ["0-90d", "91-180d", "181-365d", "366-730d", "730d+"], "recency"),
    churn_by_bucket(cust["total_orders"], [0, 1, 2, 3, 4, 1000], ["1", "2", "3", "4", "5+"], "total_orders"),
    churn_by_bucket(cust["tenure_days"], [-1, 0, 180, 365, 730, 5000], ["0 (single day)", "1-180d", "181-365d", "366-730d", "730d+"], "tenure_days"),
], ignore_index=True)
diag.to_csv(f"{TABLE_DIR}/churn_rate_by_feature.csv", index=False)
diag

,feature,bucket,churn_rate,customers
0,recency,0-90d,0.8154,10652
1,recency,91-180d,0.8069,10641
2,recency,181-365d,0.8109,19154
3,recency,366-730d,0.8059,27165
4,recency,730d+,0.8055,4457
5,total_orders,1,0.8060,35708
6,total_orders,2,0.8150,22738
7,total_orders,3,0.8081,9673
8,total_orders,4,0.8015,3008
9,total_orders,5+,0.7941,942


## 3. Train/test split and the no-skill baseline

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y)

base_rate = y_test.mean()

def evaluate(name, proba, y_true=y_test, threshold=0.5):
    pred = (proba >= threshold).astype(int)
    return {
        "Model": name,
        "ROC_AUC": round(roc_auc_score(y_true, proba), 4),
        "PR_AUC": round(average_precision_score(y_true, proba), 4),
        "Balanced_Accuracy": round(balanced_accuracy_score(y_true, pred), 4),
        "Accuracy": round(accuracy_score(y_true, pred), 4),
        "F1_churn": round(f1_score(y_true, pred), 4),
        "Retained_Recall": round(recall_score(y_true, pred, pos_label=0), 4),
    }

rows = [{
    "Model": "Baseline: predict everyone = churn",
    "ROC_AUC": 0.5, "PR_AUC": round(base_rate, 4), "Balanced_Accuracy": 0.5,
    "Accuracy": round(base_rate, 4), "F1_churn": round(2*base_rate/(1+base_rate), 4),
    "Retained_Recall": 0.0,
}]
print("Test base rate (share churned):", round(base_rate, 4))

Test base rate (share churned): 0.8088


## 4. Models (regularized, class_weight instead of SMOTE)

In [5]:
models = {
    "Logistic Regression": make_pipeline(StandardScaler(),
                                         LogisticRegression(class_weight="balanced", max_iter=1000)),
    "Decision Tree (depth 5)": DecisionTreeClassifier(max_depth=5, min_samples_leaf=200,
                                                      class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest (regularized)": RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=100,
                                                          class_weight="balanced", n_jobs=-1,
                                                          random_state=RANDOM_STATE),
}

test_proba = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    test_proba[name] = model.predict_proba(X_test)[:, 1]
    rows.append(evaluate(name, test_proba[name]))

pd.DataFrame(rows)

,Model,ROC_AUC,PR_AUC,Balanced_Accuracy,Accuracy,F1_churn,Retained_Recall
0,Baseline: predict everyone = churn,0.5000,0.8088,0.5000,0.8088,0.8943,0.0000
1,Logistic Regression,0.5082,0.8116,0.4986,0.3935,0.4670,0.6687
2,Decision Tree (depth 5),0.5004,0.8079,0.5013,0.4562,0.5603,0.5744
3,Random Forest (regularized),0.4969,0.8080,0.4974,0.5128,0.6342,0.4724


## 5. Overfitting check: your original unregularized Random Forest

In [6]:
rf_full = RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)
rf_full.fit(X_train, y_train)

overfit = pd.DataFrame([{
    "Model": "Random Forest (unpruned, as in notebook 08)",
    "Train_ROC_AUC": round(roc_auc_score(y_train, rf_full.predict_proba(X_train)[:, 1]), 4),
    "Test_ROC_AUC":  round(roc_auc_score(y_test,  rf_full.predict_proba(X_test)[:, 1]), 4),
    "Avg_tree_depth": round(np.mean([t.get_depth() for t in rf_full.estimators_]), 1),
}])
rows.append(evaluate("Random Forest (unpruned)", rf_full.predict_proba(X_test)[:, 1]))
overfit

,Model,Train_ROC_AUC,Test_ROC_AUC,Avg_tree_depth
0,"Random Forest (unpruned, as in notebook 08)",0.9987,0.4923,63.8


## 6. 5-fold cross-validated ROC-AUC (more reliable than a single split)

In [7]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring="roc_auc", n_jobs=1)
    cv_rows.append({"Model": name, "CV_ROC_AUC_mean": round(scores.mean(), 4), "CV_ROC_AUC_std": round(scores.std(), 4)})
cv_table = pd.DataFrame(cv_rows)
cv_table

,Model,CV_ROC_AUC_mean,CV_ROC_AUC_std
0,Logistic Regression,0.5053,0.0042
1,Decision Tree (depth 5),0.5053,0.0055
2,Random Forest (regularized),0.5035,0.0053


## 7. Neural network (same data, class weights, AUC reported)
Skipped automatically if TensorFlow is not installed.

In [8]:
try:
    import tensorflow as tf
    from tensorflow import keras
    tf.keras.utils.set_random_seed(RANDOM_STATE)

    scaler = StandardScaler().fit(X_train)
    Xtr, Xte = scaler.transform(X_train), scaler.transform(X_test)

    n0, n1 = (y_train == 0).sum(), (y_train == 1).sum()
    class_weight = {0: len(y_train) / (2 * n0), 1: len(y_train) / (2 * n1)}

    nn = keras.Sequential([
        keras.Input(shape=(Xtr.shape[1],)),
        keras.layers.Dense(32, activation="relu"),
        keras.layers.Dense(16, activation="relu"),
        keras.layers.Dense(1, activation="sigmoid"),
    ])
    nn.compile(optimizer="adam", loss="binary_crossentropy")
    nn.fit(Xtr, y_train, validation_split=0.2, epochs=30, batch_size=256, class_weight=class_weight,
           callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)], verbose=0)

    nn_proba = nn.predict(Xte, verbose=0).ravel()
    test_proba["Neural Network"] = nn_proba
    rows.append(evaluate("Neural Network", nn_proba))
    print("Neural network evaluated.")
except ImportError:
    print("TensorFlow not installed - neural network section skipped.")

Neural network evaluated.


## 8. Final comparison table

In [9]:
results = pd.DataFrame(rows).drop_duplicates("Model").reset_index(drop=True)
results.to_csv(f"{TABLE_DIR}/model_comparison_v2.csv", index=False)
cv_table.to_csv(f"{TABLE_DIR}/model_cv_auc_v2.csv", index=False)
results

,Model,ROC_AUC,PR_AUC,Balanced_Accuracy,Accuracy,F1_churn,Retained_Recall
0,Baseline: predict everyone = churn,0.5000,0.8088,0.5000,0.8088,0.8943,0.0000
1,Logistic Regression,0.5082,0.8116,0.4986,0.3935,0.4670,0.6687
2,Decision Tree (depth 5),0.5004,0.8079,0.5013,0.4562,0.5603,0.5744
3,Random Forest (regularized),0.4969,0.8080,0.4974,0.5128,0.6342,0.4724
4,Random Forest (unpruned),0.4923,0.8050,0.4970,0.7706,0.8689,0.0541
5,Neural Network,0.4995,0.8065,0.4965,0.5168,0.6393,0.4637


## 9. Does the model rank customers? Churn rate by predicted-risk quintile
A useful model would show clearly higher actual churn in the top quintile than the bottom quintile.

In [10]:
best_name = results[results["Model"].isin(test_proba.keys())].sort_values("ROC_AUC", ascending=False).iloc[0]["Model"]
p = pd.Series(test_proba[best_name], index=y_test.index)

quintile = pd.qcut(p.rank(method="first"), 5, labels=["Q1 (lowest score)", "Q2", "Q3", "Q4", "Q5 (highest score)"])
rank_table = (pd.DataFrame({"quintile": quintile, "actual_churn": y_test})
              .groupby("quintile", observed=True)["actual_churn"].agg(actual_churn_rate="mean", customers="count")
              .round(4).reset_index())
rank_table.insert(0, "model", best_name)
rank_table.to_csv(f"{TABLE_DIR}/risk_quintile_validation.csv", index=False)
print("Best model by held-out ROC-AUC:", best_name)
rank_table

Best model by held-out ROC-AUC: Logistic Regression


,model,quintile,actual_churn_rate,customers
0,Logistic Regression,Q1 (lowest score),0.7995,2883
1,Logistic Regression,Q2,0.8065,2883
2,Logistic Regression,Q3,0.8158,2882
3,Logistic Regression,Q4,0.8172,2883
4,Logistic Regression,Q5 (highest score),0.8051,2883


## 10. Automatic verdict

In [11]:
best_auc = results.loc[results["Model"].isin(test_proba.keys()), "ROC_AUC"].max()
cv_best = cv_table["CV_ROC_AUC_mean"].max()

print(f"Best held-out ROC-AUC : {best_auc}")
print(f"Best CV ROC-AUC       : {cv_best}")
print(f"Churn base rate       : {round(y.mean(), 4)}")

if max(best_auc, cv_best) < 0.55:
    print("\nVERDICT: No model is meaningfully better than chance (AUC < 0.55).")
    print("In this dataset churn is statistically independent of the behavioural features tested.")
    print("Do NOT present churn probabilities as predictive. Use rule-based inactivity risk instead (notebook 11).")
else:
    print("\nVERDICT: A model shows real lift. Select the best model by ROC-AUC / PR-AUC, not by plain accuracy.")

Best held-out ROC-AUC : 0.5082
Best CV ROC-AUC       : 0.5053
Churn base rate       : 0.8088

VERDICT: No model is meaningfully better than chance (AUC < 0.55).
In this dataset churn is statistically independent of the behavioural features tested.
Do NOT present churn probabilities as predictive. Use rule-based inactivity risk instead (notebook 11).


## Findings to copy into your README / report

*(Update the numbers from the tables above after you run the notebook.)*

- Churn base rate is about 81%; predicting "everyone churns" gives ~81% accuracy, so accuracy and F1 on the churn class are not informative.
- Held-out ROC-AUC for every model is about 0.50 (a coin flip); 5-fold cross-validated AUC agrees.
- The unpruned Random Forest from notebook 08 reaches ~0.999 AUC on training data but ~0.49 on test data, which is memorization of noise.
- Churn rate is ~81% in every recency, order-count and tenure bucket, and in every predicted-risk quintile.
- Conclusion: the available behavioural features carry no churn signal in this public dataset (likely synthetic). The project therefore uses transparent rule-based inactivity risk for retention actions.